In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=True)

import os
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

# Install dependencies
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "pyannote.audio==3.1.1", "pyannote.metrics==3.2.1",
    "transformers>=4.41.0", "omegaconf==2.3.0",
    "soundfile", "scikit-learn", "numpy"
], check=True)

# Patches
import sys, importlib
from types import ModuleType
from unittest.mock import MagicMock
import numpy as np
import torchaudio
import torch

np.NaN = np.nan
np.NAN = np.nan

torchaudio.set_audio_backend = lambda x: None
torchaudio.get_audio_backend  = lambda: "soundfile"

class _AudioMetaData:
    def __init__(self, sample_rate=0, num_frames=0,
                 num_channels=0, bits_per_sample=0, encoding=""):
        self.sample_rate     = sample_rate
        self.num_frames      = num_frames
        self.num_channels    = num_channels
        self.bits_per_sample = bits_per_sample
        self.encoding        = encoding

_backend_common = ModuleType("torchaudio.backend.common")
_backend_common.AudioMetaData = _AudioMetaData
_backend = ModuleType("torchaudio.backend")
_backend.common = _backend_common
sys.modules["torchaudio.backend"]        = _backend
sys.modules["torchaudio.backend.common"] = _backend_common
torchaudio.backend      = _backend
torchaudio.AudioMetaData = _AudioMetaData

if not hasattr(torchaudio, 'info'):
    def _torchaudio_info(path):
        import soundfile as sf
        info = sf.info(path)
        class _Info:
            sample_rate  = info.samplerate
            num_frames   = info.frames
            num_channels = info.channels
        return _Info()
    torchaudio.info = _torchaudio_info

sys.modules["speechbrain.pretrained"] = MagicMock()

_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

import huggingface_hub
if not getattr(huggingface_hub, '_patched_by_courtroom', False):
    _REAL_DOWNLOAD = huggingface_hub.hf_hub_download
    def _patched_download(*args, **kwargs):
        kwargs.pop("use_auth_token", None)
        return _REAL_DOWNLOAD(*args, **kwargs)
    huggingface_hub.hf_hub_download       = _patched_download
    huggingface_hub._patched_by_courtroom = True
    print("hf_hub_download patched")
else:
    _patched_download = huggingface_hub.hf_hub_download
    print("already patched")

import pyannote.audio.core.pipeline as _pp
_pp.hf_hub_download = _patched_download

import pyannote.audio
print("pyannote:", pyannote.audio.__version__)
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

Mounted at /content/drive
hf_hub_download patched
pyannote: 3.1.1
torch: 2.10.0+cu128
CUDA: True


In [ ]:
from pyannote.audio import Model
from pyannote.audio.pipelines import VoiceActivityDetection

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

vad_model = Model.from_pretrained(
    "pyannote/segmentation-3.0",
    use_auth_token=os.environ["HF_TOKEN"]
)
vad_pipeline = VoiceActivityDetection(segmentation=vad_model)
vad_pipeline.instantiate({
    "min_duration_on":  0.1,
    "min_duration_off": 0.1,
})
print("VAD loaded")

Device: cuda
VAD loaded


In [ ]:
from transformers import Wav2Vec2FeatureExtractor, WavLMForXVector
import torch

WAVLM_MODEL = "microsoft/wavlm-base-plus-sv"
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(WAVLM_MODEL)
wavlm_model       = WavLMForXVector.from_pretrained(WAVLM_MODEL).to(DEVICE)
wavlm_model.eval()
print("WavLM-base-plus-sv loaded")
print("Embedding dim:", wavlm_model.config.xvector_output_dim)

Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

WavLM-base-plus-sv loaded
Embedding dim: 512


In [ ]:
import numpy as np
import soundfile as sf
from sklearn.preprocessing import normalize
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import SpectralClustering

def extract_wavlm_embeddings(wav_path, segments, min_dur=0.3, max_chunk_s=4.0):
    audio, sr = sf.read(wav_path, dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    embeddings     = []
    valid_segments = []

    for start, end in segments:
        dur = end - start
        if dur < min_dur:
            continue

        # Split long segments into max_chunk_s chunks
        chunk_starts = np.arange(start, end, max_chunk_s)
        chunk_embeds = []

        for cs in chunk_starts:
            ce = min(cs + max_chunk_s, end)
            if ce - cs < min_dur:
                continue
            try:
                s    = int(cs * sr)
                e    = int(ce * sr)
                clip = audio[s:e]

                min_samples = int(0.5 * sr)
                if len(clip) < min_samples:
                    clip = np.pad(clip, (0, min_samples - len(clip)))

                inputs = feature_extractor(
                    clip, sampling_rate=16000,
                    return_tensors="pt", padding=True
                )
                inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

                with torch.no_grad():
                    out = wavlm_model(**inputs)
                    emb = out.embeddings.squeeze().cpu().numpy()

                chunk_embeds.append(emb)

            except Exception as ex:
                print(f"  Chunk {cs:.1f}-{ce:.1f}s skipped: {ex}")

        for j, emb in enumerate(chunk_embeds):
            chunk_start = start + j * max_chunk_s
            chunk_end   = min(chunk_start + max_chunk_s, end)
            embeddings.append(emb)
            valid_segments.append((chunk_start, chunk_end))

    if not embeddings:
        return None, []

    emb_array = np.array(embeddings).astype(np.float32)
    emb_array = normalize(emb_array, norm="l2")
    return emb_array, valid_segments

def estimate_k_spectral(affinity, max_k=20):
    degree     = affinity.sum(axis=1)
    d_inv_sqrt = np.diag(1.0 / np.sqrt(degree + 1e-8))
    laplacian  = np.eye(len(affinity)) - d_inv_sqrt @ affinity @ d_inv_sqrt
    n          = min(max_k + 1, len(affinity))
    eigs       = np.sort(np.linalg.eigvalsh(laplacian)[:n])
    gaps       = np.diff(eigs)
    k          = int(np.argmax(gaps) + 1)
    return max(1, min(k, max_k))


def cluster_embeddings(embeddings):
    if len(embeddings) == 0:
        return np.array([])
    if len(embeddings) == 1:
        return np.array([0])

    model = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=0.18,
        metric="cosine",
        linkage="average",
    )
    return model.fit_predict(embeddings)

In [ ]:
from pyannote.core import Segment, Annotation
from pyannote.metrics.diarization import DiarizationErrorRate
import csv
from pathlib import Path

AUDIO_DIR    = Path("/content/drive/MyDrive/courtroom-diarizer-eval/audio")
RTTM_DIR     = Path("/content/drive/MyDrive/courtroom-diarizer-eval/annotations/dev")
OUT_DIR      = Path("/content/drive/MyDrive/courtroom-diarizer-eval/results")
OUT_DIR.mkdir(exist_ok=True)

# ─────────────────────────────────────
BATCH        = 1      # 1, 2, 3, or 4
BATCH_SIZE   = 54
START        = (BATCH - 1) * BATCH_SIZE
END          = START + BATCH_SIZE
RESULTS_FILE = f"voxconverse_wavlm_batch{BATCH}.csv"
# ─────────────────────────────────────────────────────────────

results_path = OUT_DIR / RESULTS_FILE
existing     = []
completed    = set()
print(f"Starting WavLM batch {BATCH} fresh")

metric    = DiarizationErrorRate(collar=0.25, skip_overlap=False)
wav_files = sorted(AUDIO_DIR.glob("*.wav"))[START:END]
results   = []

print(f"Batch {BATCH}/4 — files {START+1}–{min(END,216)} ({len(wav_files)} files)")

for i, wav_path in enumerate(wav_files, START+1):
    rttm_path = RTTM_DIR / f"{wav_path.stem}.rttm"
    if not rttm_path.exists():
        continue

    try:
        # Stage 1: VAD
        vad_out  = vad_pipeline(str(wav_path))
        vad_segs = [(seg.start, seg.end)
                    for seg, _, _ in vad_out.itertracks(yield_label=True)]

        if not vad_segs:
            print(f"[{i}/216] {wav_path.stem} — no speech detected, skipping")
            continue

        # Stage 2: WavLM embeddings
        embeddings, valid_segs = extract_wavlm_embeddings(str(wav_path), vad_segs)
        if embeddings is None or len(embeddings) < 2:
            print(f"[{i}/216] {wav_path.stem} — too few segments, skipping")
            continue

        # Stage 3: Spectral clustering
        labels = cluster_embeddings(embeddings)

        # Build hypothesis annotation
        hyp = Annotation(uri=wav_path.stem)
        for (start, end), label in zip(valid_segs, labels):
            seg = Segment(start, end)
            hyp[seg, f"spk{label:02d}"] = f"spk{label:02d}"

        # Build reference annotation
        ref = Annotation(uri=wav_path.stem)
        with open(rttm_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 9 or parts[0] != "SPEAKER":
                    continue
                seg = Segment(float(parts[3]),
                              float(parts[3]) + float(parts[4]))
                ref[seg, parts[7]] = parts[7]

        # ←←← NEW: Count actual number of speakers from RTTM
        ref_speakers = len(ref.labels())

        # Compute DER
        components = metric(ref, hyp, detailed=True)
        total      = components.get("total", 0.0)
        der        = abs(metric)

        def rate(k):
            return components.get(k, 0.0) / total if total > 0 else 0.0

        info = sf.info(str(wav_path))
        row  = {
            "file":          wav_path.stem,
            "backend":       "wavlm_spectral",
            "duration_s":    round(info.duration, 2),
            "hyp_speakers":  len(set(labels)),
            "ref_speakers":  ref_speakers,
            "DER":           round(der * 100, 2),
            "miss":          round(rate("missed detection") * 100, 2),
            "false_alarm":   round(rate("false alarm") * 100, 2),
            "confusion":     round(rate("confusion") * 100, 2),
        }
        results.append(row)


        print(f"[{i}/216] {wav_path.stem} | DER={der*100:.1f}%  "
              f"K={len(set(labels))} (ref={ref_speakers})")

        # Save after every file
        fields = ["file","backend","duration_s","hyp_speakers","ref_speakers",
                  "DER","miss","false_alarm","confusion"]
        with open(results_path, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=fields)
            w.writeheader()
            w.writerows(results)

    except Exception as ex:
        print(f"[{i}] FAILED: {wav_path.stem} — {ex}")

print(f"\nBatch {BATCH} done. {len(results)} files processed.")

Starting WavLM batch 1 fresh
Batch 1/4 — files 1–54 (54 files)


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


[1/216] abjxc | DER=0.0%  K=1 (ref=1)


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


[2/216] afjiv | DER=9.1%  K=4 (ref=5)


...

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
/usr/local/lib/python3.12/dist-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


[53/216] gocbm | DER=21.6%  K=4 (ref=8)


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


[54/216] gofnj | DER=21.2%  K=2 (ref=3)

Batch 1 done. 54 files processed.


/usr/local/lib/python3.12/dist-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


In [ ]:
import csv, json
import numpy as np
from pathlib import Path

OUT_DIR = Path("/content/drive/MyDrive/courtroom-diarizer-eval/results")

# Merge WavLM batches
wavlm_results = []
for i in range(1, 5):
    p = OUT_DIR / f"voxconverse_wavlm_batch{i}.csv"
    if p.exists():
        with open(p) as f:
            rows = list(csv.DictReader(f))
            wavlm_results.extend(rows)
            print(f"WavLM batch {i}: {len(rows)} files")

# Load pyannote results
pyannote_results = []
p = OUT_DIR / "voxconverse_all_results.csv"
if p.exists():
    with open(p) as f:
        pyannote_results = list(csv.DictReader(f))
    print(f"Pyannote: {len(pyannote_results)} files")

def summarise(results, name):
    ders  = [float(r["DER"])         for r in results]
    miss  = [float(r["miss"])        for r in results]
    fa    = [float(r["false_alarm"]) for r in results]
    conf  = [float(r["confusion"])   for r in results]
    print(f"\n{'='*52}")
    print(f"  {name}")
    print(f"{'='*52}")
    print(f"  Files          : {len(results)}")
    print(f"  Mean DER       : {np.mean(ders):.2f}%")
    print(f"  Median DER     : {np.median(ders):.2f}%")
    print(f"  Std DER        : {np.std(ders):.2f}%")
    print(f"  Min / Max      : {np.min(ders):.2f}% / {np.max(ders):.2f}%")
    print(f"  Files < 10%    : {np.mean([d<10 for d in ders])*100:.1f}%")
    print(f"  Files < 20%    : {np.mean([d<20 for d in ders])*100:.1f}%")
    print(f"  Mean Miss      : {np.mean(miss):.2f}%")
    print(f"  Mean FA        : {np.mean(fa):.2f}%")
    print(f"  Mean Confusion : {np.mean(conf):.2f}%")
    return np.mean(ders)

p_der = summarise(pyannote_results, "PYANNOTE AUTO (GPU)")
w_der = summarise(wavlm_results,    "WavLM + SPECTRAL (GPU)")

print(f"\n{'='*52}")
winner = "WavLM" if w_der < p_der else "Pyannote"
print(f"  WINNER: {winner}")
print(f"  Delta:  {abs(p_der - w_der):.2f}% mean DER")
print(f"{'='*52}")

WavLM batch 1: 54 files
Pyannote: 216 files

  PYANNOTE AUTO (GPU)
  Files          : 216
  Mean DER       : 5.29%
  Median DER     : 5.34%
  Std DER        : 1.31%
  Min / Max      : 0.24% / 8.19%
  Files < 10%    : 100.0%
  Files < 20%    : 100.0%
  Mean Miss      : 1.81%
  Mean FA        : 3.01%
  Mean Confusion : 3.49%

  WavLM + SPECTRAL (GPU)
  Files          : 54
  Mean DER       : 19.55%
  Median DER     : 20.52%
  Std DER        : 3.58%
  Min / Max      : 0.00% / 22.97%
  Files < 10%    : 3.7%
  Files < 20%    : 29.6%
  Mean Miss      : 3.89%
  Mean FA        : 0.71%
  Mean Confusion : 14.94%

  WINNER: Pyannote
  Delta:  14.26% mean DER


In [ ]:
import csv, numpy as np
from pathlib import Path

OUT_DIR = Path("/content/drive/MyDrive/courtroom-diarizer-eval/results")

# Merge all 4 WavLM batches
wavlm_results = []
for i in range(1, 5):
    p = OUT_DIR / f"voxconverse_wavlm_batch{i}.csv"
    if p.exists():
        with open(p) as f:
            rows = list(csv.DictReader(f))
            wavlm_results.extend(rows)
            print(f"WavLM batch {i}: {len(rows)} files")

# Load pyannote baseline
with open(OUT_DIR / "voxconverse_all_results.csv") as f:
    pyannote_results = list(csv.DictReader(f))
print(f"Pyannote: {len(pyannote_results)} files")

def summarise(results, name):
    ders  = [float(r["DER"])         for r in results]
    miss  = [float(r["miss"])        for r in results]
    fa    = [float(r["false_alarm"]) for r in results]
    conf  = [float(r["confusion"])   for r in results]
    print(f"\n{'='*52}")
    print(f"  {name}")
    print(f"{'='*52}")
    print(f"  Files          : {len(results)}")
    print(f"  Mean DER       : {np.mean(ders):.2f}%")
    print(f"  Median DER     : {np.median(ders):.2f}%")
    print(f"  Std DER        : {np.std(ders):.2f}%")
    print(f"  Min / Max      : {np.min(ders):.2f}% / {np.max(ders):.2f}%")
    print(f"  Files < 10%    : {np.mean([d<10 for d in ders])*100:.1f}%")
    print(f"  Files < 20%    : {np.mean([d<20 for d in ders])*100:.1f}%")
    print(f"  Mean Miss      : {np.mean(miss):.2f}%")
    print(f"  Mean FA        : {np.mean(fa):.2f}%")
    print(f"  Mean Confusion : {np.mean(conf):.2f}%")
    return np.mean(ders)

p_der = summarise(pyannote_results, "PYANNOTE AUTO (GPU)")
w_der = summarise(wavlm_results,    "WavLM + AHC (GPU)")

print(f"\n{'='*52}")
winner = "WavLM" if w_der < p_der else "Pyannote"
print(f"  WINNER : {winner}")
print(f"  Delta  : {abs(p_der - w_der):.2f}% mean DER")
print(f"{'='*52}")

WavLM batch 1: 54 files
WavLM batch 2: 54 files
WavLM batch 3: 54 files
WavLM batch 4: 38 files
WavLM batch 5: 16 files
Pyannote: 216 files

  PYANNOTE AUTO (GPU)
  Files          : 216
  Mean DER       : 5.29%
  Median DER     : 5.34%
  Std DER        : 1.31%
  Min / Max      : 0.24% / 8.19%
  Files < 10%    : 100.0%
  Files < 20%    : 100.0%
  Mean Miss      : 1.81%
  Mean FA        : 3.01%
  Mean Confusion : 3.49%

  WavLM + AHC (GPU)
  Files          : 216
  Mean DER       : 22.37%
  Median DER     : 21.43%
  Std DER        : 5.40%
  Min / Max      : 0.00% / 58.92%
  Files < 10%    : 1.9%
  Files < 20%    : 23.1%
  Mean Miss      : 4.03%
  Mean FA        : 2.07%
  Mean Confusion : 15.88%

  WINNER : Pyannote
  Delta  : 17.08% mean DER
